In [27]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KernelDensity

### Membuat data baru dari dataset

In [2]:
# membaca file tsv
print("waduh")
df = pd.read_csv("../data_gempa_kaggle/katalog_gempa_v2.tsv", delimiter="\t")

waduh


C:\Users\andre\AppData\Local\Temp\ipykernel_4728\3103767649.py:3: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data_gempa_kaggle/katalog_gempa_v2.tsv", delimiter="\t")


In [8]:
# print(df.columns[15])
# df.iloc[:, 15].unique()
df = df.iloc[:, :10]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131833 entries, 0 to 131832
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   eventID      131833 non-null  object 
 1   datetime     131833 non-null  object 
 2   latitude     131833 non-null  float64
 3   longitude    131833 non-null  float64
 4   magnitude    131134 non-null  float64
 5   mag_type     131134 non-null  object 
 6   depth        131833 non-null  int64  
 7   phasecount   131831 non-null  float64
 8   azimuth_gap  131829 non-null  float64
 9   location     131817 non-null  object 
dtypes: float64(5), int64(1), object(4)
memory usage: 10.1+ MB


### Membersihkan data df utuh untuk Export

In [72]:
df.isnull().sum()

eventID          0
datetime         0
latitude         0
longitude        0
magnitude      699
mag_type       699
depth            0
phasecount       2
azimuth_gap      4
location        16
lat_grid         0
lon_grid         0
grid_id          0
dtype: int64

In [73]:
df = df.dropna(subset=["magnitude"])

df["phasecount"] = df["phasecount"].fillna(df["phasecount"].median())
df["azimuth_gap"] = df["azimuth_gap"].fillna(df["azimuth_gap"].median())

df.isnull().sum()

eventID         0
datetime        0
latitude        0
longitude       0
magnitude       0
mag_type        0
depth           0
phasecount      0
azimuth_gap     0
location       13
lat_grid        0
lon_grid        0
grid_id         0
dtype: int64

In [75]:
df['location'] = df['location'].fillna('Unknown')
df.isnull().sum()

eventID        0
datetime       0
latitude       0
longitude      0
magnitude      0
mag_type       0
depth          0
phasecount     0
azimuth_gap    0
location       0
lat_grid       0
lon_grid       0
grid_id        0
dtype: int64

In [76]:
df.to_csv("../data_gempa_kaggle/katalog_gempa_v2_cleaned.csv", index=False)

### Membuat spasial ID untuk masing-masing area wilayah
- Grid spasial adalah cara membagi permukaan bumi menjadi kotak-kotak kecil (cell/grid) berdasarkan koordinat (latitude & longitude).
- Hal ini dilakukan untuk mengubah data yang sebelumnya based on event per gempa, menjadi data per area

In [9]:
grid_size = 0.1

df["lat_grid"] = (df["latitude"] // grid_size) * grid_size
df["lon_grid"] = (df["longitude"] // grid_size) * grid_size

# Misalnya:
#   latitude = -6.234
#   grid_size = 0.1

# Hasil:
#   lat_grid = -6.3

In [10]:
df["grid_id"] = df["lat_grid"].astype(str) + "_" + df["lon_grid"].astype(str)

In [11]:
# Mengelompokkan data berdasarkan grid_id dan menghitung statistik
grid_df = df.groupby("grid_id").agg({
    "latitude": "mean",
    "longitude": "mean",
    "magnitude": ["count", "max", "mean"],
    "depth": "mean"
}).reset_index()
grid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23442 entries, 0 to 23441
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   (grid_id, )         23442 non-null  object 
 1   (latitude, mean)    23442 non-null  float64
 2   (longitude, mean)   23442 non-null  float64
 3   (magnitude, count)  23442 non-null  int64  
 4   (magnitude, max)    23368 non-null  float64
 5   (magnitude, mean)   23368 non-null  float64
 6   (depth, mean)       23442 non-null  float64
dtypes: float64(5), int64(1), object(1)
memory usage: 1.3+ MB


In [12]:
grid_df.columns = [
    "grid_id",
    "lat",
    "lon",
    "freq",
    "max_mag",
    "avg_mag",
    "avg_depth"
]

In [14]:
grid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23442 entries, 0 to 23441
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   grid_id    23442 non-null  object 
 1   lat        23442 non-null  float64
 2   lon        23442 non-null  float64
 3   freq       23442 non-null  int64  
 4   max_mag    23368 non-null  float64
 5   avg_mag    23368 non-null  float64
 6   avg_depth  23442 non-null  float64
dtypes: float64(5), int64(1), object(1)
memory usage: 1.3+ MB


### Menambahkan fitur gempa yang berada di radius 50km

In [17]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # radius bumi (km)

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [18]:
gempa_counts = []

for _, grid_row in grid_df.iterrows():
    lat_g = grid_row["lat"]
    lon_g = grid_row["lon"]

    distances = haversine(
        lat_g, lon_g,
        df["latitude"].values,
        df["longitude"].values
    )

    count = np.sum(distances <= 50)  # radius 50 km
    gempa_counts.append(count)

grid_df["gempa_in_radius_50"] = gempa_counts

#### Menambahkan fitur density (kepadatan gempa)

In [19]:
coords = df[["latitude", "longitude"]].values

In [20]:
# Train KDE
kde = KernelDensity(
    bandwidth=0.2,  # bisa dituning nanti
    kernel='gaussian'
)

kde.fit(coords)

,bandwidth,0.2
,algorithm,'auto'
,kernel,'gaussian'
,metric,'euclidean'
,atol,0
,rtol,0
,breadth_first,True
,leaf_size,40
,metric_params,None


In [21]:
# Hitung density untuk grid
grid_coords = grid_df[["lat", "lon"]].values

log_density = kde.score_samples(grid_coords)
density = np.exp(log_density)

grid_df["density"] = density

In [22]:
grid_df[["gempa_in_radius_50", "density"]].describe()

,gempa_in_radius_50,density
count,23442.000000,23442.000000
mean,315.093977,0.003836
std,374.228433,0.004824
min,1.000000,0.000030
25%,68.000000,0.000818
50%,196.000000,0.002285
75%,412.000000,0.004914
max,3137.000000,0.054300


In [30]:
grid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23442 entries, 0 to 23441
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   grid_id             23442 non-null  object 
 1   lat                 23442 non-null  float64
 2   lon                 23442 non-null  float64
 3   freq                23442 non-null  int64  
 4   max_mag             23368 non-null  float64
 5   avg_mag             23368 non-null  float64
 6   avg_depth           23442 non-null  float64
 7   gempa_in_radius_50  23442 non-null  int64  
 8   density             23442 non-null  float64
dtypes: float64(6), int64(2), object(1)
memory usage: 1.6+ MB


#### Menambahkan fitur cluster (sebagai layer ke2 selain grid sebagai grub_center)

In [24]:
# simpan sejenak. cukup untuk eksplorasi awal. nanti bisa ditambahin fitur lain, atau tuning parameter KDE, atau coba metode lain untuk density estimation.



#### Preprocessing Sedikit
- cleaning

In [31]:
# Cek jumlah null per kolom
print(grid_df.isnull().sum())

# Cek persentase null (opsional, untuk melihat signifikansi data yang hilang)
print(grid_df.isnull().mean() * 100)

grid_id                0
lat                    0
lon                    0
freq                   0
max_mag               74
avg_mag               74
avg_depth              0
gempa_in_radius_50     0
density                0
dtype: int64
grid_id               0.000000
lat                   0.000000
lon                   0.000000
freq                  0.000000
max_mag               0.315673
avg_mag               0.315673
avg_depth             0.000000
gempa_in_radius_50    0.000000
density               0.000000
dtype: float64


In [32]:
# Mengisi max_mag dengan rata-rata keseluruhan
grid_df['max_mag'] = grid_df['max_mag'].fillna(grid_df['max_mag'].mean())

# Mengisi avg_mag dengan rata-rata keseluruhan
grid_df['avg_mag'] = grid_df['avg_mag'].fillna(grid_df['avg_mag'].mean())

#### Pemilihan fitur input

In [33]:
features = [
    "freq",
    "max_mag",
    "avg_mag",
    "avg_depth",
    "gempa_in_radius_50",
    "density"
]

X = grid_df[features]

### Yang Paling Penting (DSS)
#### Labeling (Menggunakan unsupervised Clustering)
- tidak boleh menggunakan semua fitur untuk label (data leackage maseh)
- menggunakan salah satu fitur saja terlalu lemah dari sisi DSS

In [ ]:
# def label_hazard(row):
#     if row["max_mag"] >= 6 and row["density"] > 0.6:
#         return "high"
#     elif row["max_mag"] >= 5:
#         return "medium"
#     else:
#         return "low"
# grid_df["hazard"] = grid_df.apply(label_hazard, axis=1)

# memakai clustering untuk klasifikasi hazard. ini masih kasar, tapi bisa jadi starting point untuk eksplorasi lebih lanjut.
# from sklearn.cluster import KMeans

# features = ["max_mag", "freq", "density"]
# kmeans = KMeans(n_clusters=3)

# grid_df["hazard"] = kmeans.fit_predict(X)

# Opsional memakai pembobotan fitur atau scaling
grid_df["hazard_score"] = (
    grid_df["max_mag"] * 0.5 +
    grid_df["density"] * 0.3 +
    grid_df["gempa_in_radius_50"] * 0.2
)
print(grid_df[["hazard_score"]].describe())

       hazard_score
count  23442.000000
mean      65.159655
std       74.872807
min        1.841485
25%       15.698233
50%       41.222784
75%       84.552651
max      629.744739


In [57]:
q1 = grid_df["hazard_score"].quantile(0.25)
q2 = grid_df["hazard_score"].quantile(0.50)
q3 = grid_df["hazard_score"].quantile(0.75)

print(q1, q2)

15.69823326862546 41.22278445681757


In [58]:
def label_from_score(score):
    if score > q3:
        return "very_high"
    elif score > q2:
        return "high"
    elif score > q1:
        return "medium"
    else:
        return "low"
grid_df["hazard"] = grid_df["hazard_score"].apply(label_from_score)

In [71]:
# centroids = kmeans.cluster_centers_
# print(centroids)

# Mencoba export csv grid_df 
grid_df.info()
grid_export = grid_df.drop(columns=["hazard", "hazard_score"])
grid_export.head()
grid_export.to_csv("grid_df.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23442 entries, 0 to 23441
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   grid_id             23442 non-null  object 
 1   lat                 23442 non-null  float64
 2   lon                 23442 non-null  float64
 3   freq                23442 non-null  int64  
 4   max_mag             23442 non-null  float64
 5   avg_mag             23442 non-null  float64
 6   avg_depth           23442 non-null  float64
 7   gempa_in_radius_50  23442 non-null  int64  
 8   density             23442 non-null  float64
 9   hazard              23442 non-null  object 
 10  hazard_score        23442 non-null  float64
dtypes: float64(7), int64(2), object(2)
memory usage: 2.0+ MB


In [59]:
y = grid_df["hazard"]

In [36]:
# Opsional
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

#### Modeling

In [60]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = LGBMClassifier()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001396 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1369
[LightGBM] [Info] Number of data points in the train set: 18753, number of used features: 6
[LightGBM] [Info] Start training from score -1.375739
[LightGBM] [Info] Start training from score -1.378698
[LightGBM] [Info] Start training from score -1.394271
[LightGBM] [Info] Start training from score -1.396639
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
0.9965877585839198


c:\Users\andre\.conda\envs\ta-skripsi\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Feature Engineering untuk inferensi

In [40]:
# mapping lat lon menjadi grid
def get_grid(lat, lon, grid_size=0.1):
    lat_grid = (lat // grid_size) * grid_size
    lon_grid = (lon // grid_size) * grid_size
    return lat_grid, lon_grid

In [45]:
# fungsi untuk mendapatkan fitur dari lat lon
def get_nearest_features(lat, lon, grid_df):
    distances = (
        (grid_df["lat"] - lat)**2 +
        (grid_df["lon"] - lon)**2
    )
    
    idx = distances.idxmin()
    row = grid_df.loc[[idx]]  # pakai [[ ]] biar tetap 2D
    
    return row[[
        "freq",
        "max_mag",
        "avg_mag",
        "avg_depth",
        "gempa_in_radius_50",
        "density"
    ]]

In [61]:
y_pred = model.predict(X_test)
print(y_pred)

from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

['low' 'high' 'very_high' ... 'very_high' 'high' 'medium']


c:\Users\andre\.conda\envs\ta-skripsi\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.9965877585839198

In [62]:
# misalkan memakai yogyakarta 
lat = -7.8
lon = 110.4

features = get_nearest_features(lat, lon, grid_df)

print(features)
print(features.shape)

       freq   max_mag   avg_mag  avg_depth  gempa_in_radius_50   density
13525     1  2.507699  2.507699       10.0                 236  0.003303
(1, 6)


In [63]:
prediction = model.predict(features)
print(prediction)

['very_high']


In [64]:
# menyimpan model
model.booster_.save_model("hazard_model.txt")

In [ ]:
import lightgbm as lgb

model = lgb.Booster(model_file="hazzard_model.txt")